## Python notebook for generating the STAC catalog json and corresponding Item json for Raster & Vector layers

### Tools:
1. Pystac 
2. Rasterio
3. Geopandas
4. Matplotlib

This notebook returns Catalog json for Raster and Vector layers.

### 1. Importing the required modules

In [14]:
import os
import json
import xml.etree.ElementTree as ET
from datetime import datetime

import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
import pystac
from shapely.geometry import mapping, box

from pystac.extensions.raster import RasterExtension, RasterBand

In [2]:
import constants

### 2. Defining the variables used in the notebook

In [ ]:
base_dir="data/"
qml_path="data/style_file.qml"
vector_qml_file ="data/swb_style.qml"

raster_filename="saraikela-kharsawan_gobindpur_2023-07-01_2024-06-30_LULCmap_10m.tif"
vector_filename="swb2_saraikela-kharsawan_gobindpur.geojson"

corestack_dir = os.path.join(base_dir, "CorestackCatalogs")
gobindpur_dir = os.path.join(corestack_dir, "gobindpur")
raster_dir = os.path.join(gobindpur_dir, "raster")
vector_dir = os.path.join(gobindpur_dir, "vector")

os.makedirs(raster_dir, exist_ok=True)
os.makedirs(vector_dir, exist_ok=True)

raster_path = os.path.join(base_dir, raster_filename)
vector_path = os.path.join(base_dir, vector_filename)

raster_thumbnail = os.path.join(raster_dir, "raster_thumbnail.png")
vector_thumbnail = os.path.join(vector_dir, "vector_thumbnail.png")

raster_style_file = os.path.join(base_dir, "style_file.qml")
vector_style_file = os.path.join(base_dir, "swb_style.qml")


### 3. For Raster layers the data range fecthed from filename

In [5]:
def extract_raster_dates_from_filename(filename):
    try:
        print(filename)
        parts = filename.split('_')
        start_date = datetime.strptime(parts[2], "%Y-%m-%d")
        end_date = datetime.strptime(parts[3], "%Y-%m-%d")
        print(start_date)
        print(end_date)
    except Exception as e:
        raise ValueError(f"Failed to extract raster dates from filename '{filename}': {e}")
        
    return start_date, end_date    

In [6]:
extract_raster_dates_from_filename(filename=raster_filename)

saraikela-kharsawan_gobindpur_2023-07-01_2024-06-30_LULCmap_10m.tif
2023-07-01 00:00:00
2024-06-30 00:00:00


(datetime.datetime(2023, 7, 1, 0, 0), datetime.datetime(2024, 6, 30, 0, 0))

### 4. Parsing the QML file for Raster Layers

In [7]:


def parse_qml_classes(qml_path):
    tree = ET.parse(qml_path)
    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        classes.append(class_info)
    return classes

### 5. Generating the thumbnails from the files 

In [8]:
def generate_raster_thumbnail(tif_path, out_path):
    with rasterio.open(tif_path) as src:
        arr = src.read(1)
    plt.figure(figsize=(3, 3))
    plt.imshow(arr, cmap="tab20")
    plt.axis('off')
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()

def generate_vector_thumbnail(vector_path, out_path):
    gdf = gpd.read_file(vector_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)
    fig, ax = plt.subplots(figsize=(3, 3))
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")
    gdf.plot(ax=ax, color="lightblue", edgecolor="blue", linewidth=0.5)
    ax.axis('off')
    plt.savefig(out_path, dpi=150, bbox_inches='tight', pad_inches=0, facecolor=fig.get_facecolor())
    plt.close()


### 6. Creating the Raster items and adding the assets

In [23]:
try:
        start_date, end_date = extract_raster_dates_from_filename(filename=raster_filename)
except ValueError as e:
        raise RuntimeError(f"Raster item creation failed")

with rasterio.open(raster_path) as src:
    bounds = src.bounds
    geom = mapping(box(*bounds))
    bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]

generate_raster_thumbnail(raster_path, raster_thumbnail)
style_info = parse_qml_classes(raster_style_file)

print(style_info)

style_json_path = os.path.join(raster_dir, "legend.json")
with open(style_json_path, "w") as f:
    json.dump(style_info, f, indent=2)

item = pystac.Item(
    id=constants.raster_lulc_id,
    geometry=geom,
    bbox=bbox,
    datetime=start_date,
    start_datetime= start_date,
    end_datetime= end_date,
    properties={
        "title" :constants.raster_lulc_title,
        "description":constants.raster_lulc_description,
        "lulc:classes": style_info,
        
    }
)
print(item)


item.add_asset("data", pystac.Asset(
    href=f"{constants.data_url}/{raster_filename}",
    media_type=pystac.MediaType.GEOTIFF,
    roles=["data"],
    title="Raster Layer"
))

raster_ext = RasterExtension.ext(item.assets["data"], add_if_missing=True)

saraikela-kharsawan_gobindpur_2023-07-01_2024-06-30_LULCmap_10m.tif
2023-07-01 00:00:00
2024-06-30 00:00:00
[{'value': 0, 'label': 'clear', 'alpha': '0', 'color': '#000000'}, {'value': 1, 'label': 'built up', 'alpha': '255', 'color': '#ff0000'}, {'value': 2, 'label': 'kharif water', 'alpha': '255', 'color': '#74ccf4'}, {'value': 3, 'label': 'kharif and rabi water', 'alpha': '255', 'color': '#1ca3ec'}, {'value': 4, 'label': 'kharif and rabi and zaid water', 'alpha': '255', 'color': '#0f5e9c'}, {'value': 5, 'label': 'croplands', 'alpha': '255', 'color': '#f1c232'}, {'value': 6, 'label': 'Tree/Forests', 'alpha': '255', 'color': '#38761d'}, {'value': 7, 'label': 'barren lands', 'alpha': '255', 'color': '#a9a9a9'}, {'value': 8, 'label': 'Single Kharif Cropping', 'alpha': '255', 'color': '#bad93e'}, {'value': 9, 'label': 'Single Non-Kharif Cropping', 'alpha': '255', 'color': '#f59d22'}, {'value': 10, 'label': 'Double Cropping', 'alpha': '255', 'color': '#ff9371'}, {'value': 11, 'label': 'Tri

In [ ]:
raster_ext.

In [ ]:
def create_raster_item(filename):
    try:
        start_date, end_date = extract_raster_dates_from_filename(filename=raster_filename)
    except ValueError as e:
        raise RuntimeError(f"Raster item creation failed")
    
    with rasterio.open(raster_path) as src:
        bounds = src.bounds
        geom = mapping(box(*bounds))
        bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]

    generate_raster_thumbnail(raster_path, raster_thumbnail)
    style_info = parse_qml_classes(raster_style_file)

    print(style_info)
   
    style_json_path = os.path.join(raster_dir, "legend.json")
    with open(style_json_path, "w") as f:
        json.dump(style_info, f, indent=2)
  
    item = pystac.Item(
        id=constants.raster_lulc_id,
        geometry=geom,
        bbox=bbox,
        datetime=start_date,
        start_datetime= start_date,
        end_datetime= end_date,
        properties={
            "title" :constants.raster_lulc_title,
            "description":constants.raster_lulc_description,
            "lulc:classes": style_info,
            
        }
    )
    print(item)
    

    item.add_asset("data", pystac.Asset(
        href=f"{constants.data_url}/{raster_filename}",
        media_type=pystac.MediaType.GEOTIFF,
        roles=["data"],
        title="Raster Layer"
    ))

    raster_ext = RasterExtension.ext(item.assets["data"], add_if_missing=True)

    # classification_band = RasterBand.create(
    #     data_type=pystac.extensions.raster.DataType.UINT8,
    #     nodata=0,
    #     spatial_resolution=10.0
    # )
    
    # raster_ext.set_bands([classification_band])

    raster_ext.bands = [
        Band.create(data_type=DataType.UINT16),  # Example: setting data_type to UINT16
        # Add more Band objects if your raster has multiple bands
    ]    
    
    item.add_asset("thumbnail", pystac.Asset(
        href=f"{constants.base_url}/raster/thumbnail.png",
        media_type=pystac.MediaType.PNG,
        roles=["thumbnail"],
        title="Raster Thumbnail"
    ))
    item.add_asset("legend", pystac.Asset(
        href=f"{constants.base_url}/raster/legend.json",
        media_type=pystac.MediaType.JSON,
        roles=["metadata"],
        title="Legend JSON"
    ))
    item.add_asset("style", pystac.Asset(
        href=f"{constants.data_url}/../{qml_path}",
        media_type=pystac.MediaType.XML,
        roles=["metadata"],
        title="Raster style"
    ))

    item.set_self_href(os.path.join(raster_dir, "item.json"))
    item.save_object()
    return item


### 7.Creating the Vector items and adding the assets

In [20]:
def create_vector_item(vector_filename):
    start_date = constants.DEFAULT_START_DATE
    end_date = constants.DEFAULT_END_DATE

    gdf = gpd.read_file(vector_path)
    geom = mapping(gdf.union_all())
    bounds = gdf.total_bounds
    bbox = [float(b) for b in bounds]

    generate_vector_thumbnail(vector_path, vector_thumbnail)
    
    
    

    vector_metadata = {
        "number_of_records": gdf.shape[0], 
        "column_types":{col: str(dtype) for col, dtype in gdf.dtypes.items()}
    }
    
    vector_metadata_final = json.dumps(vector_metadata, indent=4)




    item = pystac.Item(
        id=constants.swb_vector_id,
        geometry=geom,
        bbox=bbox,
        datetime=start_date,
        start_datetime= start_date,
        end_datetime= end_date,
        properties={
            "title": constants.swb_vector_title,
            "description": constants.swb_vector_description,
            "vector_metadata": vector_metadata_final,
        }
    )

    print(vector_metadata_final)

    item.add_asset("data", pystac.Asset(
        href=f"{constants.data_url}/{vector_filename}",
        media_type=pystac.MediaType.GEOJSON,
        roles=["data"],
        title="Vector Layer"
    ))
    item.add_asset("thumbnail", pystac.Asset(
        href=f"{constants.base_url}/vector/thumbnail.png",
        media_type=pystac.MediaType.PNG,
        roles=["thumbnail"],
        title="Vector Thumbnail"
    ))
    item.add_asset("style", pystac.Asset(
        href=f"{constants.data_url}/../{vector_style_file}",
        media_type=pystac.MediaType.XML,
        roles=["style"],
        title="Vector style"
    ))
    
    item.set_self_href(os.path.join(vector_dir, "item.json"))
    item.save_object()
    return item


In [21]:
vector_item=create_vector_item(vector_filename)
print(vector_item)


{
    "number_of_records": 2306,
    "column_types": {
        "id": "object",
        "MWS_UID": "object",
        "UID": "object",
        "any": "int32",
        "area_17-18": "float64",
        "area_18-19": "float64",
        "area_19-20": "float64",
        "area_20-21": "float64",
        "area_21-22": "float64",
        "area_22-23": "float64",
        "area_23-24": "float64",
        "area_ored": "float64",
        "category_sq_m": "object",
        "k_17-18": "float64",
        "k_18-19": "float64",
        "k_19-20": "float64",
        "k_20-21": "float64",
        "k_21-22": "float64",
        "k_22-23": "float64",
        "k_23-24": "float64",
        "kr_17-18": "float64",
        "kr_18-19": "float64",
        "kr_19-20": "float64",
        "kr_20-21": "float64",
        "kr_21-22": "float64",
        "kr_22-23": "float64",
        "kr_23-24": "float64",
        "krz_17-18": "float64",
        "krz_18-19": "float64",
        "krz_19-20": "float64",
        "krz_20-21": "

### 8. Using the raster and vector items created and generating the Catalog 

In [22]:
catalog = pystac.Catalog(
    id=constants.id_main,
    title=constants.title_main,
    description=constants.description_main
)
catalog.add_item(create_raster_item(filename=raster_filename))
catalog.add_item(create_vector_item(vector_filename=vector_filename))
catalog.set_self_href(os.path.join(gobindpur_dir, "catalog.json"))

corestack_catalog = pystac.Catalog(
    id="corestack",
    title="CorestackCatalogs",
    description="Root catalog containing all subcatalogs"
)
corestack_catalog.add_child(catalog)
corestack_catalog.set_self_href(os.path.join(corestack_dir, "catalog.json"))
corestack_catalog.normalize_and_save(corestack_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)

print(f" Root STAC Catalog created at: {os.path.join(corestack_dir, 'catalog.json')}")


saraikela-kharsawan_gobindpur_2023-07-01_2024-06-30_LULCmap_10m.tif
2023-07-01 00:00:00
2024-06-30 00:00:00
[{'value': 0, 'label': 'clear', 'alpha': '0', 'color': '#000000'}, {'value': 1, 'label': 'built up', 'alpha': '255', 'color': '#ff0000'}, {'value': 2, 'label': 'kharif water', 'alpha': '255', 'color': '#74ccf4'}, {'value': 3, 'label': 'kharif and rabi water', 'alpha': '255', 'color': '#1ca3ec'}, {'value': 4, 'label': 'kharif and rabi and zaid water', 'alpha': '255', 'color': '#0f5e9c'}, {'value': 5, 'label': 'croplands', 'alpha': '255', 'color': '#f1c232'}, {'value': 6, 'label': 'Tree/Forests', 'alpha': '255', 'color': '#38761d'}, {'value': 7, 'label': 'barren lands', 'alpha': '255', 'color': '#a9a9a9'}, {'value': 8, 'label': 'Single Kharif Cropping', 'alpha': '255', 'color': '#bad93e'}, {'value': 9, 'label': 'Single Non-Kharif Cropping', 'alpha': '255', 'color': '#f59d22'}, {'value': 10, 'label': 'Double Cropping', 'alpha': '255', 'color': '#ff9371'}, {'value': 11, 'label': 'Tri

AttributeError: 'AssetRasterExtension' object has no attribute 'set_bands'

In [ ]:
import pystac
from pystac.extensions.raster import RasterExtension, RasterBand
from pystac.extensions.projection import ProjectionExtension
from datetime import datetime
import json

def create_raster_stac_catalog():
    """
    Create a STAC catalog with raster extension items including data type and nodata values.
    """
    
    # Create the main catalog
    catalog = pystac.Catalog(
        id="raster-catalog",
        description="Sample raster data catalog with raster extension",
        title="Raster Data Catalog"
    )
    
    # Create a collection for satellite imagery
    collection = pystac.Collection(
        id="satellite-imagery",
        description="Satellite imagery collection with raster metadata",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([[-180, -90, 180, 90]]),
            temporal=pystac.TemporalExtent([[datetime(2023, 1, 1), datetime(2023, 12, 31)]])
        ),
        title="Satellite Imagery Collection",
        license="CC-BY-4.0"
    )
    
    # Add the collection to the catalog
    catalog.add_child(collection)
    
    # Create sample raster items with different data types and configurations
    items = []
    
    # Item 1: Landsat-like multispectral imagery
    item1 = create_multispectral_item()
    items.append(item1)
    
    # Item 2: Digital Elevation Model
    item2 = create_dem_item()
    items.append(item2)
    
    # Item 3: Land Cover Classification
    item3 = create_landcover_item()
    items.append(item3)
    
    # Add items to collection
    for item in items:
        collection.add_item(item)
    
    return catalog

def create_multispectral_item():
    """Create a multispectral imagery item with raster extension."""
    
    # Create the item
    item = pystac.Item(
        id="landsat-sample-001",
        geometry={
            "type": "Polygon",
            "coordinates": [[
                [-122.5, 37.7],
                [-122.3, 37.7],
                [-122.3, 37.9],
                [-122.5, 37.9],
                [-122.5, 37.7]
            ]]
        },
        bbox=[-122.5, 37.7, -122.3, 37.9],
        datetime=datetime(2023, 6, 15),
        properties={
            "platform": "landsat-8",
            "instruments": ["oli", "tirs"]
        }
    )
    
    # Add assets with raster extension
    # Red band
    red_asset = pystac.Asset(
        href="https://example.com/landsat/red.tif",
        media_type=pystac.MediaType.GEOTIFF,
        title="Red Band",
        description="Landsat 8 Red Band (Band 4)"
    )
    item.add_asset("red", red_asset)
    
    # Green band
    green_asset = pystac.Asset(
        href="https://example.com/landsat/green.tif",
        media_type=pystac.MediaType.GEOTIFF,
        title="Green Band",
        description="Landsat 8 Green Band (Band 3)"
    )
    item.add_asset("green", green_asset)
    
    # Blue band
    blue_asset = pystac.Asset(
        href="https://example.com/landsat/blue.tif",
        media_type=pystac.MediaType.GEOTIFF,
        title="Blue Band",
        description="Landsat 8 Blue Band (Band 2)"
    )
    item.add_asset("blue", blue_asset)
    
    # NIR band
    nir_asset = pystac.Asset(
        href="https://example.com/landsat/nir.tif",
        media_type=pystac.MediaType.GEOTIFF,
        title="Near Infrared Band",
        description="Landsat 8 NIR Band (Band 5)"
    )
    item.add_asset("nir", nir_asset)
    
    # Configure raster bands for each asset
    raster_bands = {
        "red": [RasterBand.create(
            data_type=pystac.extensions.raster.DataType.UINT16,
            nodata=0,
            spatial_resolution=30.0,
            statistics={"minimum": 0, "maximum": 65535, "mean": 8500.5, "stddev": 2100.3}
        )],
        "green": [RasterBand.create(
            data_type=pystac.extensions.raster.DataType.UINT16,
            nodata=0,
            spatial_resolution=30.0,
            statistics={"minimum": 0, "maximum": 65535, "mean": 9200.1, "stddev": 1950.7}
        )],
        "blue": [RasterBand.create(
            data_type=pystac.extensions.raster.DataType.UINT16,
            nodata=0,
            spatial_resolution=30.0,
            statistics={"minimum": 0, "maximum": 65535, "mean": 7800.9, "stddev": 1800.2}
        )],
        "nir": [RasterBand.create(
            data_type=pystac.extensions.raster.DataType.UINT16,
            nodata=0,
            spatial_resolution=30.0,
            statistics={"minimum": 0, "maximum": 65535, "mean": 12500.3, "stddev": 3200.8}
        )]
    }
    
    # Apply raster extension to each asset
    for asset_key, bands in raster_bands.items():
        raster_ext = RasterExtension.ext(item.assets[asset_key], add_if_missing=True)
        raster_ext.set_bands(bands)
    
    # Add projection extension
    proj_ext = ProjectionExtension.ext(item, add_if_missing=True)
    proj_ext.epsg = 4326
    proj_ext.shape = [7801, 7681]
    proj_ext.transform = [30.0, 0.0, -122.5, 0.0, -30.0, 37.9, 0.0, 0.0, 1.0]
    
    return item

def create_dem_item():
    """Create a Digital Elevation Model item with raster extension."""
    
    item = pystac.Item(
        id="dem-sample-001",
        geometry={
            "type": "Polygon",
            "coordinates": [[
                [-105.2, 40.0],
                [-105.0, 40.0],
                [-105.0, 40.2],
                [-105.2, 40.2],
                [-105.2, 40.0]
            ]]
        },
        bbox=[-105.2, 40.0, -105.0, 40.2],
        datetime=datetime(2023, 8, 20),
        properties={
            "platform": "srtm",
            "gsd": 30.0
        }
    )
    
    # Add DEM asset
    dem_asset = pystac.Asset(
        href="https://example.com/dem/elevation.tif",
        media_type=pystac.MediaType.GEOTIFF,
        title="Digital Elevation Model",
        description="SRTM 30m DEM"
    )
    item.add_asset("elevation", dem_asset)
    
    # Apply raster extension to the asset
    raster_ext = RasterExtension.ext(item.assets["elevation"], add_if_missing=True)
    
    # Configure raster band for elevation data
    elevation_band = RasterBand.create(
        data_type=pystac.extensions.raster.DataType.FLOAT32,
        nodata=-9999.0,
        spatial_resolution=30.0,
        unit="meter",
        statistics={
            "minimum": 1580.2,
            "maximum": 4401.8,
            "mean": 2890.5,
            "stddev": 650.3
        }
    )
    
    raster_ext.set_bands([elevation_band])
    
    # Add projection extension
    proj_ext = ProjectionExtension.ext(item, add_if_missing=True)
    proj_ext.epsg = 4326
    proj_ext.shape = [2400, 2400]
    proj_ext.transform = [0.000833333, 0.0, -105.2, 0.0, -0.000833333, 40.2, 0.0, 0.0, 1.0]
    
    return item

def create_landcover_item():
    """Create a land cover classification item with raster extension."""
    
    item = pystac.Item(
        id="landcover-sample-001",
        geometry={
            "type": "Polygon",
            "coordinates": [[
                [-120.0, 35.0],
                [-119.0, 35.0],
                [-119.0, 36.0],
                [-120.0, 36.0],
                [-120.0, 35.0]
            ]]
        },
        bbox=[-120.0, 35.0, -119.0, 36.0],
        datetime=datetime(2023, 4, 10),
        properties={
            "platform": "modis",
            "classification_scheme": "IGBP"
        }
    )
    
    # Add land cover asset
    lc_asset = pystac.Asset(
        href="https://example.com/landcover/classification.tif",
        media_type=pystac.MediaType.GEOTIFF,
        title="Land Cover Classification",
        description="MODIS Land Cover Type"
    )
    item.add_asset("classification", lc_asset)
    
    # Apply raster extension to the asset
    raster_ext = RasterExtension.ext(item.assets["classification"], add_if_missing=True)
    
    # Configure raster band for classification data
    classification_band = RasterBand.create(
        data_type=pystac.extensions.raster.DataType.UINT8,
        nodata=255,
        spatial_resolution=500.0,
        statistics={
            "minimum": 1,
            "maximum": 17,
            "mean": 8.5,
            "stddev": 4.2
        }
    )
    
    raster_ext.set_bands([classification_band])
    
    # Add projection extension
    proj_ext = ProjectionExtension.ext(item, add_if_missing=True)
    proj_ext.epsg = 4326
    proj_ext.shape = [240, 240]
    proj_ext.transform = [0.00416667, 0.0, -120.0, 0.0, -0.00416667, 36.0, 0.0, 0.0, 1.0]
    
    return item

def save_catalog(catalog, output_dir="./stac_catalog"):
    """Save the STAC catalog to disk."""
    import os
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Normalize and save the catalog
    catalog.normalize_and_save(output_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
    
    print(f"STAC catalog saved to: {output_dir}")
    print(f"Catalog root: {os.path.join(output_dir, 'catalog.json')}")

def print_catalog_info(catalog):
    """Print information about the created catalog."""
    print("=== STAC Catalog Information ===")
    print(f"Catalog ID: {catalog.id}")
    print(f"Description: {catalog.description}")
    print(f"Number of collections: {len(list(catalog.get_collections()))}")
    
    for collection in catalog.get_collections():
        print(f"\nCollection: {collection.id}")
        print(f"  Description: {collection.description}")
        print(f"  Number of items: {len(list(collection.get_items()))}")
        
        for item in collection.get_items():
            print(f"\n  Item: {item.id}")
            print(f"    Platform: {item.properties.get('platform', 'N/A')}")
            print(f"    Date: {item.datetime}")
            print(f"    Assets: {list(item.assets.keys())}")
            
            # Print raster information if available
            for asset_key, asset in item.assets.items():
                if RasterExtension.has_extension(asset):
                    raster_ext = RasterExtension.ext(asset)
                    bands = raster_ext.get_bands()
                    if bands:
                        band = bands[0]  # Get first band info
                        print(f"      {asset_key}: {band.data_type}, nodata={band.nodata}")

def main():
    """Main function to create and save the STAC catalog."""
    
    # Create the catalog
    catalog = create_raster_stac_catalog()
    
    # Print catalog information
    print_catalog_info(catalog)
    
    # Validate the catalog
    print("\n=== Validation ===")
    try:
        catalog.validate()
        print("✓ Catalog is valid!")
    except Exception as e:
        print(f"✗ Catalog validation failed: {e}")
    
    # Save the catalog
    print("\n=== Saving Catalog ===")
    save_catalog(catalog)
    
    # Print JSON representation of first item for inspection
    print("\n=== Sample Item JSON ===")
    first_item = next(iter(catalog.get_items()))
    print(json.dumps(first_item.to_dict(), indent=2))

if __name__ == "__main__":
    main()